# Notebook 02: Chunking Strategy, Dense Embeddings & FAISS Indexing
### Project: Enterprise Document Intelligence & RAG Assistant
**Objective:**
1. Benchmark recursive semantic chunking with configurable overlap
2. Generate 384-dimensional dense embeddings using `all-MiniLM-L6-v2`
3. Build FAISS vector database and measure cosine similarity search retrieval accuracy.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from src.document_loader import DocumentLoader
from src.chunking import DocumentChunker
from src.embeddings import EmbeddingManager
from src.retriever import FAISSRetriever

# Load documents
loader = DocumentLoader()
docs = loader.load_directory('../data/documents')
print(f"Loaded {len(docs)} documents.")

### 1. Chunking Analysis: Evaluating Chunk Sizes and Overlap
We test chunk sizes of 300, 500, and 800 characters to observe chunk count and sentence boundary preservation.

In [ ]:
for size in [300, 500, 800]:
    chunker = DocumentChunker(chunk_size=size, chunk_overlap=int(size * 0.12))
    test_chunks = chunker.chunk_documents(docs)
    avg_len = sum(len(c.text) for c in test_chunks) / max(1, len(test_chunks))
    print(f"Config: Size={size} Overlap={int(size*0.12)} -> Chunks Produced: {len(test_chunks)}, Avg Char Length: {avg_len:.1f}")

### 2. Sentence-Transformers Dense Embedding Generation
We use `all-MiniLM-L6-v2` to produce unit-normalized 384-dimensional vector embeddings.

In [ ]:
chunker = DocumentChunker(chunk_size=500, chunk_overlap=60)
chunks = chunker.chunk_documents(docs)
print(f"Selected baseline chunks: {len(chunks)}")

embed_mgr = EmbeddingManager(model_name="all-MiniLM-L6-v2")
chunk_texts = [c.text for c in chunks]
vectors = embed_mgr.embed_texts(chunk_texts, normalize=True)
print("Vector Matrix Shape:", vectors.shape)
print("Embedding Dimension:", embed_mgr.embedding_dim)

### 3. FAISS Vector Database Construction & Semantic Search

In [ ]:
retriever = FAISSRetriever(embed_mgr, index_dir="../vectorstore")
indexed_count = retriever.build_index(chunks, force_rebuild=True)
print(f"FAISS index built and saved. Total indexed vectors: {indexed_count}")

# Execute test query
query = "What is the notice period for resignation?"
results = retriever.retrieve(query, top_k=3)

print(f"\nQuery: '{query}'")
for idx, r in enumerate(results, 1):
    print(f"\n--- Result #{idx} (Cosine Similarity: {r.score:.4f}) ---")
    print(f"Source: {r.filename} | Page {r.page_number} | Section: {r.section}")
    print("Text:", r.text)